In [1]:
import numpy as np
import pandas as pd
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from groq import Groq
from PyPDF2 import PdfReader
import faiss, os, time

d:\Projects\RAG Q&A App\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_pdf(path):
    reader = PdfReader(path)
    return "".join([p.extract_text() for p in reader.pages if p.extract_text()])

docs = {
    "menu":     read_pdf("test.pdf"),
    "research": read_pdf("test1.pdf"),
    "resume":   read_pdf("test2.pdf")
}

configs = [
    {"splitter": "CharacterTextSplitter",          "chunk_size": 500,  "overlap": 50},
    {"splitter": "CharacterTextSplitter",          "chunk_size": 1000, "overlap": 100},
    {"splitter": "CharacterTextSplitter",          "chunk_size": 1500, "overlap": 150},
    {"splitter": "RecursiveCharacterTextSplitter", "chunk_size": 500,  "overlap": 50},
    {"splitter": "RecursiveCharacterTextSplitter", "chunk_size": 1000, "overlap": 100},
]

In [3]:
queries_per_doc = {
    "menu": [
        "What is the main topic of this document?",
        "What food categories are available?",
        "What are the most expensive items?",
        "What are the ordering or delivery notes?",
        "Which items are seasonal or available on request?"
    ],
    "research": [
        "What is the main topic of this document?",
        "What are the key findings or contributions?",
        "Who are the authors?",
        "What NLP methodology was used?",
        "What are the limitations of the proposed chatbot?"
    ],
    "resume": [
        "What is the main topic of this document?",
        "What are the key skills mentioned?",
        "What is the person's educational background?",
        "What projects has the person worked on?",
        "What certifications or publications are mentioned?"
    ]
}

In [ ]:
from secret_api_keys import groq_api_key

client = Groq(api_key=groq_api_key)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": False}
)

results = []

for doc_name, doc_text in docs.items():
    test_queries = queries_per_doc[doc_name]
    
    for cfg in configs:
        if cfg["splitter"] == "CharacterTextSplitter":
            splitter = CharacterTextSplitter(
                chunk_size=cfg["chunk_size"], chunk_overlap=cfg["overlap"])
        else:
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=cfg["chunk_size"], chunk_overlap=cfg["overlap"])

        chunks = splitter.split_text(doc_text)

        sample = np.array(embeddings.embed_query("sample"))
        index = faiss.IndexFlatL2(sample.shape[0])
        vs = FAISS(
            embedding_function=embeddings,
            index=index,
            docstore=InMemoryDocstore(),
            index_to_docstore_id={}
        )
        vs.add_texts(chunks)

        for query in test_queries:
            start = time.time()
            docs_retrieved = vs.similarity_search(query, k=4)
            context = "\n\n".join([d.page_content for d in docs_retrieved])
            latency = round(time.time() - start, 3)

            import time

            for attempt in range(5):
                try:
                    completion = client.chat.completions.create(
                        messages=[{"role": "user", "content":
                            f"Answer based ONLY on context.\nContext:{context[:2000]}\nQuestion:{query}"}],
                        model="llama-3.3-70b-versatile",
                        temperature=0.3, max_tokens=150
                    )
                    answer = completion.choices[0].message.content.strip()
                    break
                except Exception as e:
                    print(f"Rate limit hit, waiting 60s... (attempt {attempt+1})")
                    time.sleep(60)

            results.append({
                "doc": doc_name,
                "splitter": cfg["splitter"],
                "chunk_size": cfg["chunk_size"],
                "overlap": cfg["overlap"],
                "num_chunks": len(chunks),
                "query": query,
                "answer": answer,
                "retrieval_latency_s": latency,
                "score": None
            })

d:\Projects\RAG Q&A App\.venv\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Rate limit hit, waiting 60s... (attempt 1)
Rate limit hit, waiting 60s... (attempt 2)
Rate limit hit, waiting 60s... (attempt 3)


In [ ]:
import os
if os.path.exists("chunking_evaluation.csv"):
    os.remove("chunking_evaluation.csv")

df = pd.DataFrame(results)
df.to_csv("chunking_evaluation.csv", index=False)
print(df[["doc", "splitter", "chunk_size", "num_chunks", "query", "retrieval_latency_s"]])

In [ ]:
df = pd.read_csv("chunking_evaluation.csv")

# Average score per config per doc
summary = df.groupby(["doc", "splitter", "chunk_size"])["score"].mean().reset_index()
summary.columns = ["doc", "splitter", "chunk_size", "avg_score"]
summary = summary.sort_values(["doc", "avg_score"], ascending=[True, False])
print(summary)

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv("chunking_evaluation.csv")

# Summary
summary = df.groupby(["doc", "splitter", "chunk_size"])["score"].mean().reset_index()
summary.columns = ["doc", "splitter", "chunk_size", "avg_score"]
summary = summary.sort_values(["doc", "avg_score"], ascending=[True, False])
print(summary)

# Bar chart
docs_list = summary["doc"].unique()
fig, axes = plt.subplots(1, len(docs_list), figsize=(15, 5), sharey=True)

for ax, doc in zip(axes, docs_list):
    data = summary[summary["doc"] == doc]
    labels = data["splitter"].str.replace("CharacterTextSplitter", "CTS").str.replace("RecursiveCharacterTextSplitter", "RCTS") + "\n" + data["chunk_size"].astype(str)
    ax.bar(labels, data["avg_score"], color=["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"])
    ax.set_title(doc)
    ax.set_ylim(0, 3)
    ax.set_ylabel("Avg Score")
    ax.set_xlabel("Config")
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle("Chunking Config Performance by Document", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("chunking_evaluation_chart.png", dpi=150)
plt.show()
print("Chart saved!")